In [0]:
%run ../../utils/utils

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import pyspark.sql.functions as F

##  Ler as 3 tabelas Gold consolidadas

In [0]:
pdf_regras = ler_delta("gold", "gold_dq_regras_consolidado", STORAGE_OPTIONS).toPandas()
pdf_saude_hora = ler_delta("gold", "gold_dq_saude_consolidado", STORAGE_OPTIONS).toPandas()
pdf_saude_pct = ler_delta("gold", "gold_saude_percentual_consolidado", STORAGE_OPTIONS).toPandas()
 
print(f"Linhas em gold_dq_regras_consolidado: {len(pdf_regras)}")
print(f"Linhas em gold_dq_saude_consolidado: {len(pdf_saude_hora)}")
print(f"Linhas em gold_saude_percentual_consolidado: {len(pdf_saude_pct)}")
 

##  Resumo geral — saúde de todas as tabelas lado a lado

In [0]:
resumo = pdf_saude_pct[[
    "tabela", "total_bronze_distintos", "total_quarentena",
    "percentual_saude", "percentual_invalidado"
]].sort_values("percentual_invalidado", ascending=False)
 
print("===== RESUMO GERAL DE SAÚDE POR TABELA =====")
display(resumo)
 
pior_tabela = resumo.iloc[0]
melhor_tabela = resumo.iloc[-1]
print(f"\nPior saúde: {pior_tabela['tabela']} ({pior_tabela['percentual_invalidado']}% invalidado)")
print(f"Melhor saúde: {melhor_tabela['tabela']} ({melhor_tabela['percentual_invalidado']}% invalidado)")

##  Ranking de saúde por tabela (gráfico)

In [0]:
resumo_ordenado = resumo.sort_values("percentual_saude", ascending=True)
 
fig, ax = plt.subplots(figsize=(10, max(4, len(resumo_ordenado) * 0.6)))
cores = ["#C44E52" if v < 80 else "#DD8452" if v < 95 else "#55A868" for v in resumo_ordenado["percentual_saude"]]
ax.barh(resumo_ordenado["tabela"], resumo_ordenado["percentual_saude"], color=cores)
ax.set_xlabel("% Saúde (passou para a Silver)")
ax.set_title("Ranking de Saúde por Tabela")
ax.set_xlim(0, 100)
for i, v in enumerate(resumo_ordenado["percentual_saude"]):
    ax.text(v + 1, i, f"{v}%", va="center", fontweight="bold")
plt.tight_layout()
plt.show()

##  Top 10 regras com mais falhas (somado em todas as tabelas e datas)

In [0]:
if not pdf_regras.empty:
    top_regras = (
        pdf_regras.groupby(["tabela", "regra", "severidade"])["total_falhas"]
        .sum()
        .reset_index()
        .sort_values("total_falhas", ascending=False)
        .head(10)
    )
 
    print("===== TOP 10 REGRAS COM MAIS FALHAS (todas as tabelas) =====")
    display(top_regras)
 
    fig, ax = plt.subplots(figsize=(10, 6))
    rotulos = top_regras["tabela"] + " | " + top_regras["regra"]
    cores_sev = ["#C44E52" if s == "Critica" else "#DD8452" for s in top_regras["severidade"]]
    ax.barh(rotulos[::-1], top_regras["total_falhas"][::-1], color=cores_sev[::-1])
    ax.set_xlabel("Total de falhas")
    ax.set_title("Top 10 Regras com Mais Falhas (todas as tabelas)")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum log de falha encontrado em nenhuma tabela.")

##  Falhas por tabela, separadas por severidade (Crítica vs Aviso)

In [0]:
if not pdf_regras.empty:
    falhas_por_tabela_severidade = (
        pdf_regras.groupby(["tabela", "severidade"])["total_falhas"]
        .sum()
        .unstack(fill_value=0)
    )
 
    print("===== FALHAS POR TABELA E SEVERIDADE =====")
    display(falhas_por_tabela_severidade.reset_index())
 
    falhas_por_tabela_severidade.plot(
        kind="barh", stacked=True, figsize=(10, max(4, len(falhas_por_tabela_severidade) * 0.6)),
        color={"Critica": "#C44E52", "Aviso": "#DD8452"}
    )
    plt.xlabel("Total de falhas")
    plt.title("Falhas por Tabela (Crítica vs Aviso)")
    plt.tight_layout()
    plt.show()

##  Volume de registros limpos ao longo do tempo, por tabela

In [0]:
if not pdf_saude_hora.empty:
    pdf_saude_hora["data_hora"] = pd.to_datetime(pdf_saude_hora["data_hora"])
 
    fig, ax = plt.subplots(figsize=(12, 6))
    for tabela, grupo in pdf_saude_hora.groupby("tabela"):
        grupo_ordenado = grupo.sort_values("data_hora")
        ax.plot(grupo_ordenado["data_hora"], grupo_ordenado["qtd_limpos"], marker="o", label=tabela)
 
    ax.set_xlabel("Data/Hora")
    ax.set_ylabel("Registros limpos (Silver)")
    ax.set_title("Volume de Registros Limpos ao Longo do Tempo, por Tabela")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum dado de saúde por hora encontrado.")

##  Alertas — tabelas que precisam de atenção

In [0]:
LIMIAR_ATENCAO = 80.0  # % de saúde abaixo disso vira alerta
 
tabelas_criticas = resumo[resumo["percentual_saude"] < LIMIAR_ATENCAO]
 
if not tabelas_criticas.empty:
    print(f"⚠️  {len(tabelas_criticas)} tabela(s) abaixo de {LIMIAR_ATENCAO}% de saúde:\n")
    for _, row in tabelas_criticas.iterrows():
        print(f"  - {row['tabela']}: {row['percentual_saude']}% saudável "
              f"({row['total_quarentena']} de {row['total_bronze_distintos']} invalidados)")
else:
    print(f"✅ Todas as tabelas estão acima de {LIMIAR_ATENCAO}% de saúde.")